$$
\huge \text{Model Training and Experiment Tracking}
$$

In this notebook, we train, evaluate, and compare different models for our movie recommendation system. We will test:
1. **Collaborative Filtering** (Matrix Factorization using SGD and SVD)
2. **Content-Based Filtering** (using movie tags and metadata)
3. **Hybrid Models** (combining both approaches)

We use **MLflow** to log our hyperparameters and metrics, making it easy to track and compare all our experiments.

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import os
import sys
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

# Add the parent directory to the system path to import local modules
sys.path.append(os.path.abspath(".."))
PROJECT_ROOT = Path().resolve().parent

# Import custom functions and models from our src folder
from src.features import build_content_matrix, cf_to_numpy, merge_movies_with_tags, prepare_cf_data
from src.models import (
    MF_SGD,
    MF_SVD,
    cold_start_flow,
    content_based_recommendations,
    evaluate_hit_rate,
    evaluate_hit_rate_fast,
    hybrid_model_recommendations,
    make_hybrid_evaluator,
    train_test_split_per_user,
    user_content_recommendations,
)

# Set up MLflow tracking
mlflow.set_tracking_uri("http://localhost:5000")

## 1. Data
We load the processed data and format it. Content-based models work directly with DataFrames, while Collaborative Filtering models require the data to be converted into NumPy arrays.

In [4]:
movies = pd.read_csv("../data/processed/movies.csv")
tags =  pd.read_csv("../data/processed/tags.csv")
ratings =  pd.read_csv("../data/processed/ratings.csv")

In [5]:
cf_ratings = ratings.copy()
cf_movies = movies.copy()

# Data splits and user/movie maps for Collaborative Filtering
cf_train_df, cf_test_df, cf_user_map, cf_movie_map = prepare_cf_data(cf_ratings)
(cf_train_users, cf_train_items, cf_train_ratings, cf_test_users, cf_test_items, cf_test_ratings) = cf_to_numpy(cf_train_df, cf_test_df)

## 2. Collaborative Filtering Models

### 2.1 Matrix Factorization using Stochastic Gradient Descent (SGD)
This model learns user and item preferences by minimizing rating errors iteratively. We log the hyperparameters and the metrics (RMSE, MAE) directly to MLflow.

In [ ]:
mlflow.set_experiment("Collaborative_Filtering_Methods") 

configs = [ {"k": 20, "lr": 0.01, "reg": 0.02, "epochs": 3}, 
           {"k": 50, "lr": 0.005, "reg": 0.01, "epochs": 5}] 

for cfg in configs: 
    run_name = f"SGD_k{cfg['k']}_lr{cfg['lr']}_reg{cfg['reg']}"
    with mlflow.start_run(run_name=run_name): 
        mlflow.set_tags({ "model_type": "SGD", "family": "matrix_factorization", "impl": "sgd" }) 
        mlflow.log_params({**cfg, "model_type": "SGD" })

        model = MF_SGD( k=cfg["k"], lr=cfg["lr"], reg=cfg["reg"], epochs=cfg["epochs"] ) 
        model.fit( cf_train_users, cf_train_items, cf_train_ratings, len(cf_user_map), len(cf_movie_map) ) 
        preds = model.predict(cf_test_users, cf_test_items) 
        
        # metrics
        rmse = np.sqrt(np.mean((cf_test_ratings - preds) ** 2)) 
        mae = np.mean(np.abs(cf_test_ratings - preds)) 
        mlflow.log_metrics({"rmse": rmse, "mae": mae}) 

        mlflow.sklearn.log_model(sk_model=model, name="sgd_model")

#### SGD Results
Below are the Top 10 movies recommended by the trained SGD model for our target user $1$.

In [ ]:
# == SGD model Loading from MLflow ==
# params: k=50, lr=0.005, reg=0.01, epochs=5
model_sgd_uri = "models:/m-a9e3f846b1644dc8a1af08509922038c"
loaded_model_sgd = mlflow.sklearn.load_model(model_sgd_uri)
target_user_id = 1

rec_sgd = loaded_model_sgd.recommend(
    user_id=target_user_id,
    user_map=cf_user_map,
    movie_map=cf_movie_map,
    df_movies=movies,
    top_n=10
)

print(f"Top 10 recommendations for user {target_user_id}:")
rec_sgd

Top 10 recommendations for user 1:


,title
291,Pulp Fiction (1994)
4860,"Lord of the Rings: The Fellowship of the Ring,..."
5811,"Lord of the Rings: The Two Towers, The (2002)"
40864,Planet Earth (2006)
42734,Over the Garden Wall (2013)
45982,Band of Brothers (2001)
46131,Planet Earth II (2016)
46366,Cosmos
60969,Parasite (2019)
68338,Twelve Angry Men (1954)


### 2.2 Matrix Factorization using Singular Value Decomposition (SVD)
As an alternative approach, we train a classic SVD model to decompose the user-item ratings matrix.

In [ ]:
mlflow.set_experiment("Collaborative_Filtering_Methods") 

configs = [ {"k": 10}, {"k": 20}, {"k": 50}, {"k": 100}, {"k": 150}]

for cfg in configs:
    run_name = f"SVD_k{cfg['k']}"
    with mlflow.start_run(run_name=run_name): 
        mlflow.set_tags({ "model_type": "SVD", "family": "matrix_factorization", "impl": "svd" })
        mlflow.log_params({**cfg, "model_type": "SVD" })

        model = MF_SVD(k=cfg["k"])
        model.fit(cf_train_users, cf_train_items, cf_train_ratings, len(cf_user_map), len(cf_movie_map))
        preds = model.predict(cf_test_users, cf_test_items)

        # metrics
        rmse = np.sqrt(np.mean((cf_test_ratings - preds) ** 2))        
        mae = np.mean(np.abs(cf_test_ratings - preds))
        mlflow.log_metrics({"rmse": rmse, "mae": mae})

        mlflow.sklearn.log_model(sk_model=model, name="svd_model")

#### SVD Results 
Below are the Top 10 movies recommended by the classic SVD model for our target user $1$:

In [8]:
# == SVD model Loading from MLflow ==
# params: k=50
model_svd_uri = "models:/m-28ddb72cddf44bf3ab5a1c37d4038597"
loaded_model_svd = mlflow.sklearn.load_model(model_svd_uri)
target_user_id = 1

rec_svd = loaded_model_svd.recommend(
    user_id=target_user_id,
    user_map=cf_user_map,
    movie_map=cf_movie_map,
    df_movies=movies,
    top_n=10
)

print(f"Top 10 recommendations for user {target_user_id}:")
rec_svd

Top 10 recommendations for user 1:


,title
256,Star Wars: Episode IV - A New Hope (1977)
534,Blade Runner (1982)
759,Independence Day (a.k.a. ID4) (1996)
834,"Godfather, The (1972)"
897,2001: A Space Odyssey (1968)
1161,Aliens (1986)
1174,Alien (1979)
1181,"Godfather: Part II, The (1974)"
1198,"Terminator, The (1984)"
2742,American Beauty (1999)


### 2.3 Comparative Analysis: SGD vs SVD Recommendations

We can observe that **SGD** and **SVD** generate different Top 10 recommendation lists for the same user. This is an expected outcome in collaborative filtering, as the two architectures use fundamentally different training mechanisms to fill in the missing ratings:

1. Different Optimization Objectives
   * **SVD** is a classic matrix factorization approach that looks at the overall structure of the dataset. Because of this, it tends to prioritize universally popular, highly-rated historical classics (e.g., *Star Wars*, *The Godfather*), making its recommendations lean toward established "all-time favorites."
   * **SGD** learns iteratively by updating parameters based on individual user-item interactions. This localized learning process allows it to pick up on different patterns, surfacing modern hits, high-quality contemporary series, and distinct community trends (e.g., *Parasite*, *Band of Brothers*).

2. Algorithmic Variance
   * Even though both models map users and movies into a shared latent space, the differences in hyperparameter configurations (such as regularization and learning rates in SGD) naturally lead to different final embedding representations.

## 3. Content-Based Recommendation
This model recommends movies based on their features (genres and tags) and how well they match what a user has watched before. We evaluate this model using the **Hit Rate@10** metric.

### 3.1 Merging movie metadata with tags

The first step was to enrich the movie dataset with additional descriptive information. The original `movies` dataset contains movie identifiers, titles, and genres, while the `tags` dataset contains user-generated textual tags associated with movies.

To create a richer representation of movie content, both datasets were merged using the `movieId` attribute. Since a movie can have multiple tags assigned by different users, all tags belonging to the same movie were aggregated into a single text field.

The resulting dataset (`movies_plus_tags`) contains both genre information and descriptive tags, which provide additional semantic information about movie characteristics. This enriched representation serves as the foundation for the content-based recommendation model.


In [6]:
movies_plus_tags = merge_movies_with_tags(movies, tags)
movies_plus_tags.head()

,movieId,title,genres,tag
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,children Disney animation children Disney Disn...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Robin Williams fantasy Robin Williams time tra...
2,3,Grumpier Old Men (1995),Comedy|Romance,comedinha de velhinhos engraÃƒÂ§ada comedinha ...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,characters slurs based on novel or book chick ...
4,5,Father of the Bride Part II (1995),Comedy,Fantasy pregnancy remake family Steve Martin s...


### 3.2 Content-Based Recommendation for a Single Movie
To generate content-based recommendations, a TF-IDF (Term Frequency–Inverse Document Frequency) representation was created from movie genres and tags.

Genres separated by the "|" character were converted into individual terms, and then combined with movie tags to create a textual description of each movie. The TF-IDF algorithm transformed these descriptions into numerical vectors representing movie content.

After constructing the TF-IDF matrix, cosine similarity was used to measure the similarity between movies. For a selected movie, the system identifies the most similar movies based on the angle between their TF-IDF vectors.

In this example, recommendations were generated for the movie "Toy Story (1995)".

In [5]:
matrix, vectorizer = build_content_matrix(movies_plus_tags)
list_of_recomendation_films_content_based = content_based_recommendations(1, movies_plus_tags, matrix, top_n=10)
list_of_recomendation_films_content_based

,movieId,title
2997,3114,Toy Story 2 (1999)
2243,2355,"Bug's Life, A (1998)"
14761,78499,Toy Story 3 (2010)
4753,4886,"Monsters, Inc. (2001)"
39729,157296,Finding Dory (2016)
6229,6377,Finding Nemo (2003)
8213,8961,"Incredibles, The (2004)"
26515,126405,The Adventures of André and Wally B. (1984)
19789,103141,Monsters University (2013)
18236,95856,Knick Knack (1989)


The obtained recommendations appear highly relevant. Most recommended movies belong to the same animation and family-oriented genre as Toy Story. Several recommendations are also Pixar productions, including Finding Nemo, Finding Dory, The Incredibles, and Monsters, Inc.

This result suggests that the TF-IDF representation successfully captured important content characteristics such as animation, family themes, children's entertainment, and studio-specific patterns reflected in tags.

The recommendations demonstrate that the content-based approach can effectively identify movies with similar thematic and stylistic properties.

### 3.3 Content-Based Recommendations for the User
While the previous experiment generated recommendations from a single movie, the next step was to build recommendations based on a user's entire viewing history.

A user profile was constructed by combining the TF-IDF vectors of movies positively rated by the user. The profile vector represents the user's preferences and interests. Recommendations were then generated by calculating cosine similarity between the user profile and all available movies in the dataset.

This approach allows the system to recommend movies that are similar to the overall preferences of the user rather than to a single title.


In [6]:
target_user_id = 1

list_of_recomendation_films_content_based_user = user_content_recommendations(
    user_id=target_user_id,
    ratings=ratings,
    movies=movies_plus_tags,
    matrix=matrix,
    top_n=10
)
list_of_recomendation_films_content_based_user

,movieId,title
893,920,Gone with the Wind (1939)
1208,1250,"Bridge on the River Kwai, The (1957)"
7148,7303,The Diary of Anne Frank (1959)
1830,1936,Mrs. Miniver (1942)
3355,3475,"Place in the Sun, A (1951)"
1821,1927,All Quiet on the Western Front (1930)
834,858,"Godfather, The (1972)"
1827,1933,"Life of Emile Zola, The (1937)"
1829,1935,How Green Was My Valley (1941)
1251,1293,Gandhi (1982)


The recommendations suggest that the user has a strong preference for historical, biographical, war, and classic drama films. Many recommended movies are critically acclaimed classics that share similar themes and narrative styles.

Unlike the previous example, the recommendations are not limited to a single genre. Instead, they reflect broader patterns extracted from the user's complete viewing history.

This demonstrates one of the main advantages of content-based filtering: the ability to build personalized recommendations tailored to individual preferences.

### 3.4 Evaluation using Hit Rate@10
To evaluate the recommendation system, the rating history of each user was divided into training and test subsets using an 80/20 split.

For every user, 80% of rated movies were used to construct the user profile, while the remaining 20% served as unseen test data. Recommendations were generated using only the training portion of the user's history.

The Hit Rate@10 metric was then calculated. A recommendation was considered successful if at least one movie from the user's test set appeared among the top 10 recommended movies.

Hit Rate@10 is defined as the proportion of users for whom at least one relevant movie was successfully recommended within the first 10 positions.

In [ ]:
# Split data
ratings_train, ratings_test = train_test_split_per_user(ratings)

# Evaluation
score = evaluate_hit_rate_fast(
    model_func=user_content_recommendations,
    ratings_train=ratings_train,
    ratings_test=ratings_test,
    movies=movies_plus_tags,
    matrix=matrix,
    k=10
)

print(f"Hit Rate@10: {score:.4f}")

  0%|          | 0/200948 [00:00<?, ?it/s]

Hit Rate@10: 0.5323


The obtained Hit Rate@10 value of 0.5323 indicates that the system successfully recommended at least one relevant movie within the top 10 recommendations for approximately 53.2% of users.

Considering the large size of the movie catalog (over 87,000 movies), this result can be considered satisfactory. The probability of randomly recommending a movie from a user's future viewing history is extremely low. Therefore, achieving a hit rate above 50% suggests that the model captures meaningful information about user preferences.

The results confirm that combining genres and tags with TF-IDF representations can produce effective content-based recommendations.

### 4.5 General Conclusions
The content-based recommendation system successfully utilized movie genres and user-generated tags to identify similarities between movies and generate personalized recommendations.

Experiments demonstrated that the model was capable of recommending highly relevant movies both for individual titles and for complete user profiles. The evaluation using Hit Rate@10 yielded a score of 0.5323, indicating that the system was able to recover relevant items for more than half of the evaluated users.

## 4. Hybrid Recommendation Model
To get the best of both worlds, we implement a hybrid model. It combines the scores from Collaborative Filtering (SGD) and Content-Based filtering using a weight parameter ($\alpha$).

In [7]:
matrix, vectorizer = build_content_matrix(movies_plus_tags)
target_user_id = 1
alpha = 0.5

In [9]:
k_sgd = 20
epochs = 3
lr = 0.01
reg = 0.02

sgd_model = MF_SGD(k=k_sgd, epochs=epochs, lr=lr, reg=reg)
sgd_model.fit(cf_train_users, cf_train_items, cf_train_ratings, len(cf_user_map), len(cf_movie_map))

[cf-sgd] Epoch 1: RMSE = 0.8896
[cf-sgd] Epoch 2: RMSE = 0.8544
[cf-sgd] Epoch 3: RMSE = 0.8287


In [24]:
hybrid_results = hybrid_model_recommendations(
    user_id=target_user_id, 
    ratings=ratings, 
    movies=movies_plus_tags, 
    matrix=matrix, 
    cf_model=sgd_model, 
    user_map=cf_user_map, 
    movie_map=cf_movie_map, 
    alpha=alpha, 
    top_n=10
)

hybrid_results

,movieId,title
1208,1250,"Bridge on the River Kwai, The (1957)"
893,920,Gone with the Wind (1939)
834,858,"Godfather, The (1972)"
3355,3475,"Place in the Sun, A (1951)"
1821,1927,All Quiet on the Western Front (1930)
1829,1935,How Green Was My Valley (1941)
895,922,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)
1840,1946,Marty (1955)
887,914,My Fair Lady (1964)
1251,1293,Gandhi (1982)


In [ ]:
ratings_train, ratings_test = train_test_split_per_user(ratings)

print(type(sgd_model))

hybrid_eval = make_hybrid_evaluator(
    cf_model=sgd_model,
    user_map=cf_user_map,
    movie_map=cf_movie_map,
    alpha=0.5
)

# Evaluation
score = evaluate_hit_rate_fast(
    model_func=hybrid_eval,
    ratings_train=ratings_train,
    ratings_test=ratings_test,
    movies=movies_plus_tags,
    matrix=matrix,
    k=10
)

print(f"Hit Rate@10: {score:.4f}")

<class 'src.models.MF_SGD'>


100%|██████████| 200948/200948 [6:06:49<00:00,  9.13it/s]    

Hit Rate@10: 0.6002


## 5. Handling the Cold Start Problem
When a brand-new user joins the platform, we don't have their history. This interactive flow asks the new user to rate a few popular movies, and then uses the Content-Based model to give them their first recommendations instantly.

### 5.1 Introduction
One of the most important challenges in recommender systems is the cold start problem. This issue occurs when a new user enters the system and no historical interaction data is available. Since recommendation algorithms typically rely on previous ratings or viewing history, generating personalized recommendations for a completely new user becomes difficult.

To address this problem, an interactive onboarding procedure was implemented. Instead of requiring an existing rating history, the system asks the user to rate a predefined set of popular and widely known movies.

### 5.2 Methodology
The implemented solution follows the steps below:
1. A list of popular movies is presented to the user.
2. The user rates each movie on a scale from 0 to 5, including half-point ratings.
3. Ratings equal to zero indicate that the movie has not been watched.
4. A new user profile is created and assigned a unique user identifier.
5. The collected ratings are appended to the ratings dataset.
6. Content-based recommendations are generated for each positively rated movie.
7. A personalized user profile is constructed from all positively rated movies.
8. The final personalized recommendations are generated using the user-based content recommendation model.

If the user does not provide any positive ratings, a fallback strategy based on a predefined list of popular movies is applied to initialize the recommendation process.

In [ ]:
ratings_path = PROJECT_ROOT / "data" / "processed" / "ratings.csv"

new_user_id = cold_start_flow(
    ratings,
    movies,
    movies_plus_tags,
    matrix,
    content_based_recommendations,
    user_content_recommendations,
    ratings_path=ratings_path
)


    Did you like this movie?

    1 = strongly dislike
    2 = dislike
    3 = neutral
    4 = like
    5 = strongly like
    0 = not seen

    You can also use half ratings (e.g. 1.5, 2.5, 3.5, 4.5)
    
Invalid rating. Try again (0,1,1.5,...,5)

Saved 14 ratings for new user.

--- CONTENT-BASED RECOMMENDATIONS ---

Since you like the movie: Toy Story (1995)
       movieId                                        title
2997      3114                           Toy Story 2 (1999)
2243      2355                         Bug's Life, A (1998)
14761    78499                           Toy Story 3 (2010)
4753      4886                        Monsters, Inc. (2001)
39729   157296                          Finding Dory (2016)
6229      6377                          Finding Nemo (2003)
8213      8961                      Incredibles, The (2004)
26515   126405  The Adventures of André and Wally B. (1984)
19789   103141                   Monsters University (2013)
18236    95856                       

### 5.3 Results

The generated recommendations indicate that the system successfully captured user preferences despite having no prior interaction history.

For example, after the user expressed positive opinions about Toy Story (1995), the system recommended several highly related animated movies, including Toy Story 2, Toy Story 3, Finding Nemo, Finding Dory, Monsters, Inc., and The Incredibles. These recommendations share similar genres, themes, and production characteristics.

Similarly, positive ratings for GoldenEye (1995) resulted in recommendations dominated by James Bond movies such as Goldfinger, Dr. No, Casino Royale, and Tomorrow Never Dies. This demonstrates that the content representation effectively captured franchise-specific information.

The recommendations generated for Forrest Gump (1994) included The Green Mile, Saving Private Ryan, Cast Away, and Philadelphia, which share similar dramatic and historical elements. Likewise, movies recommended for Jurassic Park (1993) were strongly related to dinosaurs, adventure, and science-fiction themes.

##### Analysis of Personalized Recommendations

Examining the final personalized recommendations, it can be observed that the system did not focus exclusively on a single genre or type of movie. During the onboarding process, the user expressed positive preferences for animated films such as *Toy Story*, adventure movies such as *Jurassic Park*, historical and action-oriented productions such as *Gladiator*, dramatic films such as *Forrest Gump*, and romantic dramas such as *Titanic*.

As a result, the generated user profile became more diverse and represented multiple areas of interest simultaneously. This diversity is reflected in the final recommendations, which include animated movies (*Toy Story 2*, *Toy Story 3*), historical and war dramas (*Saving Private Ryan*), character-driven dramas (*Road to Perdition*), classic cinema (*Gone with the Wind*), fantasy adventures (*The Lord of the Rings: The Return of the King*), and biographical or historical stories (*Captain Phillips*).

This behavior indicates that the recommendation model successfully aggregates information from the user's entire rating history rather than relying on a single movie or genre. Consequently, the generated recommendations provide a more comprehensive representation of the user's preferences and are likely to be more relevant than recommendations based solely on individual items.

These examples suggest that the TF-IDF representation built from genres and tags successfully identified meaningful semantic relationships between movies.

### 5.4 Conclusions
The implemented cold start strategy successfully addressed the lack of historical user data by collecting a small number of initial ratings during the onboarding process.

Even with a limited number of rated movies, the system was able to generate meaningful recommendations that reflected the user's interests. The obtained recommendations were generally consistent with the genres, themes, franchises, and narrative styles of the positively rated movies.

The experiment demonstrates that combining an interactive onboarding procedure with a content-based recommendation model provides an effective solution to the cold start problem.